# Bridge II Lab: Probability, Loss, and Optimization

In this lab, we explore core probability and optimization concepts for machine learning: Bayes' theorem, the sigmoid function, log-loss, and gradient descent. Each section builds intuition through computation and visualization.

## Setup

This notebook uses numpy for numerical operations and matplotlib for visualization. All code is self-contained and reproducible with fixed random seeds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(42)

## Part 1: Bayes' Theorem in Practice

We build a student support classifier: given a message contains an urgent keyword, what is the probability the student is genuinely in crisis? We compute the posterior probability and visualize how the prior affects the result.

In [ ]:
# Student crisis support: Bayes' theorem
# Event U: student is in crisis
# Event K: message contains urgent keyword

# Known probabilities
p_urgent = 0.10                    # P(urgent) = 10% of students are in crisis
p_keyword_given_urgent = 0.80      # P(keyword|urgent) = 80% of crisis students use keyword
p_keyword_given_not_urgent = 0.15  # P(keyword|not urgent) = 15% of non-crisis use keyword

# Compute P(keyword) using the law of total probability
p_keyword = (p_keyword_given_urgent * p_urgent + 
             p_keyword_given_not_urgent * (1 - p_urgent))

# Compute P(urgent|keyword) using Bayes' theorem
p_urgent_given_keyword = (p_keyword_given_urgent * p_urgent) / p_keyword

print("Bayes' Theorem for Student Crisis Detection")
print("="*50)
print(f"P(urgent)                   = {p_urgent:.3f}")
print(f"P(keyword|urgent)           = {p_keyword_given_urgent:.3f}")
print(f"P(keyword|not urgent)       = {p_keyword_given_not_urgent:.3f}")
print()
print(f"P(keyword) [total prob]     = {p_keyword:.3f}")
print(f"P(urgent|keyword) [Bayes]   = {p_urgent_given_keyword:.3f}")
print()
print(f"Interpretation:")
print(f"  If a message contains the urgent keyword,")
print(f"  there is a {p_urgent_given_keyword*100:.1f}% chance the student is truly in crisis.")

In [ ]:
# Vary the prior and observe how the posterior changes
priors = np.linspace(0.01, 0.50, 50)
posteriors = []

for prior in priors:
    p_kw = (p_keyword_given_urgent * prior + 
            p_keyword_given_not_urgent * (1 - prior))
    posterior = (p_keyword_given_urgent * prior) / p_kw
    posteriors.append(posterior)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(priors * 100, np.array(posteriors) * 100, linewidth=2.5, color='steelblue')
ax.scatter([p_urgent * 100], [p_urgent_given_keyword * 100], 
          s=150, color='coral', zorder=5, label='Our scenario')

ax.set_xlabel('Prior P(urgent) [%]', fontsize=12)
ax.set_ylabel('Posterior P(urgent|keyword) [%]', fontsize=12)
ax.set_title('How Prior Belief Affects Posterior Probability', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("Observation:")
print("  If we increase the base rate (prior) of crises,")
print("  the posterior probability of crisis given the keyword also increases.")
print("  This shows why base rates matter!")

## Part 2: The Sigmoid Function

We plot the sigmoid function, which maps any real number to the probability range (0, 1). We show how scaling the input affects the slope.

In [ ]:
# Define sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Define step function for comparison
def step_function(z):
    return (z > 0).astype(float)

# Generate input range
z = np.linspace(-6, 6, 1000)

# Compute sigmoid and step outputs
sig = sigmoid(z)
step = step_function(z)

# Plot sigmoid and step function
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(z, sig, linewidth=2.5, color='steelblue', label='Sigmoid: 1/(1+exp(-z))')
ax.plot(z, step, linewidth=2.5, color='coral', linestyle='--', label='Step function (threshold at 0)')

# Mark special points
ax.scatter([0], [0.5], s=100, color='steelblue', zorder=5)
ax.text(0.3, 0.55, 'sigmoid(0)=0.5', fontsize=10)

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)

ax.set_xlim(-6, 6)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Input z', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('Sigmoid Function vs. Step Function', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

print("Sigmoid properties:")
print(f"  sigmoid(-6) ≈ {sigmoid(-6):.4f}")
print(f"  sigmoid(0)  = {sigmoid(0):.4f}")
print(f"  sigmoid(6)  ≈ {sigmoid(6):.4f}")
print()
print("The sigmoid is smooth and differentiable, making it ideal for gradient-based learning.")

In [ ]:
# Show effect of scaling the input
scales = [0.5, 1.0, 2.0]
colors_scale = ['lightblue', 'steelblue', 'darkblue']

fig, ax = plt.subplots(figsize=(10, 6))

for scale, color in zip(scales, colors_scale):
    sig_scaled = sigmoid(scale * z)
    ax.plot(z, sig_scaled, linewidth=2.5, color=color, label=f'sigmoid({scale}*z)')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)

ax.set_xlim(-6, 6)
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('Input z', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('Effect of Input Scaling on Sigmoid Slope', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.show()

print("Observation:")
print("  Larger scale values make the sigmoid steeper.")
print("  Scale = 0.5: gradual transition (soft confidence)")
print("  Scale = 2.0: sharp transition (high confidence)")

## Part 3: Log-Loss by Hand

We compute log-loss for a set of predictions and show why confident wrong predictions are heavily penalized.

In [ ]:
# Log-loss (binary cross-entropy) for a single example with y=1
# L(y_pred) = -[y * log(y_pred) + (1-y) * log(1-y_pred)]
# For y=1: L(y_pred) = -log(y_pred)

def log_loss(y_true, y_pred):
    """Compute binary cross-entropy loss."""
    epsilon = 1e-15  # Avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Example: true label = 1 (positive example)
y_true = 1
predictions = np.array([0.9, 0.6, 0.3, 0.1])

print("Log-Loss for True Label = 1")
print("="*50)
for pred in predictions:
    loss = log_loss(y_true, pred)
    print(f"  Prediction: {pred:.1f} -> Loss: {loss:.3f}")
print()
print("Observation:")
print("  - Correct predictions (0.9) have low loss")
print("  - Incorrect predictions (0.1) have high loss")
print("  - The loss increases exponentially for wrong confident predictions")

In [ ]:
# Plot log-loss curve
y_pred_range = np.linspace(0.01, 0.99, 1000)
losses_y1 = log_loss(1, y_pred_range)
losses_y0 = log_loss(0, y_pred_range)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(y_pred_range, losses_y1, linewidth=2.5, color='steelblue', label='Loss when y=1')
ax.plot(y_pred_range, losses_y0, linewidth=2.5, color='coral', label='Loss when y=0')

# Mark our example predictions for y=1
for pred in predictions:
    loss = log_loss(y_true, pred)
    ax.scatter([pred], [loss], s=100, color='steelblue', zorder=5)
    ax.text(pred, loss + 0.3, f'{pred:.1f}', fontsize=9, ha='center')

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Log-Loss', fontsize=12)
ax.set_title('Log-Loss (Binary Cross-Entropy)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_ylim(0, 5)

plt.tight_layout()
plt.show()

## Part 4: Gradient Descent on a 1D Loss

We define a nonconvex loss function and implement gradient descent with different learning rates. We visualize the loss surface and optimization trajectories.

In [ ]:
# Define a 1D loss function: L(w) = (w-3)^2 + 0.5*(w-3)^4
# This has a global minimum at w=3

def loss_fn(w):
    return (w - 3)**2 + 0.5 * (w - 3)**4

def loss_gradient(w):
    """Gradient of L(w) with respect to w."""
    return 2 * (w - 3) + 2 * (w - 3)**3

# Gradient descent with different learning rates
def gradient_descent(w_init, learning_rate, n_steps=100):
    """Run gradient descent."""
    w = w_init
    trajectory = [w]
    losses = [loss_fn(w)]
    
    for _ in range(n_steps):
        grad = loss_gradient(w)
        w = w - learning_rate * grad
        trajectory.append(w)
        losses.append(loss_fn(w))
    
    return np.array(trajectory), np.array(losses)

# Initial weight
w_init = 0.0

# Run gradient descent with different learning rates
learning_rates = [0.01, 0.1, 0.5]
colors = ['lightblue', 'steelblue', 'darkblue']
results = {}

for lr, color in zip(learning_rates, colors):
    traj, losses = gradient_descent(w_init, lr)
    results[lr] = (traj, losses, color)
    print(f"Learning rate {lr:.2f}: w converges to {traj[-1]:.4f}, final loss {losses[-1]:.4f}")

In [ ]:
# Plot the loss surface and optimization trajectories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss surface with trajectories
w_range = np.linspace(-1, 5, 1000)
loss_range = loss_fn(w_range)

ax1.plot(w_range, loss_range, linewidth=2.5, color='black', label='Loss function', zorder=2)
ax1.scatter([3], [0], s=150, color='red', marker='*', zorder=5, label='Global minimum')

for lr, (traj, losses, color) in results.items():
    ax1.plot(traj[:20], losses[:20], 'o-', linewidth=2, color=color, 
            markersize=6, alpha=0.7, label=f'lr={lr}')

ax1.set_xlabel('Weight w', fontsize=12)
ax1.set_ylabel('Loss L(w)', fontsize=12)
ax1.set_title('Gradient Descent Trajectories on Loss Surface', fontsize=13)
ax1.set_ylim(0, 15)
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Plot 2: Loss over iterations
for lr, (traj, losses, color) in results.items():
    ax2.plot(losses, linewidth=2.5, color=color, label=f'lr={lr}')

ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Training Loss Over Iterations', fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("\nObservation:")
print("  lr=0.01: Slow but stable convergence")
print("  lr=0.1:  Good balance of speed and stability")
print("  lr=0.5:  Diverges (overshoots the minimum)")

## Part 5: Full Training Loop - Logistic Regression

We generate synthetic data from a logistic model, implement logistic regression from scratch, and train it using gradient descent.

In [ ]:
# Generate synthetic data from a logistic model
# y ~ Bernoulli(sigmoid(2*x - 1))

np.random.seed(42)
n_samples = 50

# Generate X values
X = np.random.uniform(-1, 2, n_samples)

# Generate y based on the true logistic function: P(y=1|x) = sigmoid(2*x - 1)
true_logit = 2 * X - 1
true_prob = sigmoid(true_logit)
y = (np.random.rand(n_samples) < true_prob).astype(int)

print(f"Generated {n_samples} samples from logistic model")
print(f"True weights: w=2, b=-1")

In [ ]:
# Implement logistic regression from scratch

class LogisticRegressor:
    def __init__(self, learning_rate=0.1, n_steps=200):
        self.learning_rate = learning_rate
        self.n_steps = n_steps
        self.w = None
        self.b = None
        self.loss_history = []
    
    def forward(self, X):
        """Compute predictions: P(y=1|x) = sigmoid(w*x + b)."""
        logit = self.w * X + self.b
        return sigmoid(logit)
    
    def compute_loss(self, X, y):
        """Compute mean log-loss over all samples."""
        y_pred = self.forward(X)
        return np.mean(log_loss(y, y_pred))
    
    def backward(self, X, y):
        """Compute gradients of loss with respect to w and b."""
        y_pred = self.forward(X)
        error = y_pred - y
        
        grad_w = np.mean(error * X)
        grad_b = np.mean(error)
        
        return grad_w, grad_b
    
    def fit(self, X, y):
        """Train the model."""
        self.w = 0.0
        self.b = 0.0
        
        for step in range(self.n_steps):
            loss = self.compute_loss(X, y)
            self.loss_history.append(loss)
            
            grad_w, grad_b = self.backward(X, y)
            
            self.w -= self.learning_rate * grad_w
            self.b -= self.learning_rate * grad_b
            
            if (step + 1) % 50 == 0:
                print(f"Step {step+1}: loss={loss:.4f}, w={self.w:.4f}, b={self.b:.4f}")
    
    def predict(self, X):
        """Make predictions."""
        return self.forward(X)

# Train logistic regression
model = LogisticRegressor(learning_rate=0.5, n_steps=200)
model.fit(X, y)

print(f"\nFinal weights: w={model.w:.4f}, b={model.b:.4f}")
print(f"True weights:  w=2.0000, b=-1.0000")

In [ ]:
# Plot training loss curve and final decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss over iterations
ax1.plot(model.loss_history, linewidth=2.5, color='steelblue')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Over Iterations', fontsize=13)
ax1.grid(True, alpha=0.3)

# Plot 2: Data points and learned decision boundary
ax2.scatter(X[y == 0], y[y == 0], c='red', s=80, alpha=0.6, edgecolors='darkred', label='y=0')
ax2.scatter(X[y == 1], y[y == 1], c='blue', s=80, alpha=0.6, edgecolors='darkblue', label='y=1')

# Plot the learned probability curve
X_range = np.linspace(-1.5, 2.5, 1000)
y_pred = model.predict(X_range)
ax2.plot(X_range, y_pred, linewidth=2.5, color='steelblue', label='Learned P(y=1|x)')

# Plot the true probability curve
y_true_prob = sigmoid(2 * X_range - 1)
ax2.plot(X_range, y_true_prob, linewidth=2.5, color='green', linestyle='--', label='True P(y=1|x)')

# Mark decision boundary (where p=0.5)
boundary_x = -model.b / model.w
ax2.axvline(x=boundary_x, color='coral', linestyle=':', linewidth=2, label='Decision boundary')

ax2.set_xlabel('Feature X', fontsize=12)
ax2.set_ylabel('Probability / Label', fontsize=12)
ax2.set_title('Logistic Regression: Data and Decision Boundary', fontsize=13)
ax2.set_ylim(-0.1, 1.2)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10, loc='upper left')

plt.tight_layout()
plt.show()

## What to Notice

**Bayes' Theorem:**
- The posterior probability depends on both the likelihood and the prior
- Base rates matter: a high prior makes rare events more likely given evidence
- Bayes' theorem is the foundation for probabilistic inference and decision-making

**Sigmoid Function:**
- Maps any real input to (0, 1), ideal for probability estimation
- Smooth and differentiable, enabling gradient-based learning
- Input scaling controls confidence: larger coefficients make sharper transitions

**Log-Loss:**
- Measures disagreement between predicted and true probabilities
- Heavily penalizes confident wrong predictions (e.g., predicting 0.1 when y=1)
- Differs from accuracy: gives credit to almost correct predictions

**Gradient Descent:**
- Learning rate controls step size: too small is slow, too large diverges
- Each update moves in the direction opposite to the gradient
- Converges to local minima (may not be global for nonconvex losses)

**Logistic Regression:**
- Models P(y=1|x) = sigmoid(w*x + b)
- Trained by minimizing log-loss
- Learned boundary matches the true underlying process when sufficient data is available